In [20]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib as plt
import seaborn as sn 
from pykalman import KalmanFilter
# from ... import 
# from ... import 
# from ... import 

# Scripts 
from utils.backtest import *
from utils.cointegration import *
from utils.kalman import *
from utils.pairs_det import *
from utils.pairs import *

# Import data <br>
- Create algorithm to compute cointegrated pairs (pairs-det.py)
- Do human check on the pairs that the algorithm chose by plotting pairs and computing simple EDA

In [ ]:
# 收集数据来自 Bloomberg 然后将这些对整理到 x 和 y 字典中。 

# Run Cointegration test
*Create the algorithm to run stationarity test, differenced regression, Engle-Granger test and estimate the ECM* (cointegration.py)

For each equity pair, I first test whether both series are integrated of order one, I(1), using ADF-based checks. If both satisfy this condition, I estimate the long-run relationship using OLS and then test the residuals for stationarity following the Engle–Granger two-step approach. When residuals are stationary, I conclude that the pair is cointegrated and proceed to estimate an Error Correction Model (ECM).

In [ ]:
# 在继续进行协整检验之前，请确定 x 和 y 的字典。

# Integration order checks
y_order_res = determine_integration_order(y, max_diff=max_diff, regression=regression)
x_order_res = determine_integration_order(x, max_diff=max_diff, regression=regression)

y_order = y_order_res["order"]
x_order = x_order_res["order"]

results["asset_y_integration"] = y_order_res
results["asset_x_integration"] = x_order_res
results["same_order_integration"] = (y_order == x_order)

# Standard Engle-Granger setup: both series should be I(1)
if y_order is None or x_order is None:
    results["decision"] = "Integration order could not be determined"
    return results

if y_order != 1 or x_order != 1:
    results["decision"] = "Both series are not I(1), so standard Engle-Granger is not appropriate"
    return results

In [ ]:
# Run OLS in levels
ols_model, residuals = run_ols(y, x)

hedge_ratio = ols_model.params.get("x", np.nan)
intercept = ols_model.params.get("const", np.nan)

results["ols_summary"] = {
     "params": ols_model.params.to_dict(),
    "pvalues": ols_model.pvalues.to_dict(),
    "rsquared": ols_model.rsquared
}

# Store 
results["hedge_ratio"] = hedge_ratio
results["intercept"] = intercept
results["spread"] = residuals
results["y_aligned"] = y
results["x_aligned"] = x

# Residual stationarity test (Engle-Granger step 2)
resid_adf = adf_test(residuals, regression="c")
results["residual_adf"] = resid_adf

if not resid_adf["is_stationary_5pct"]:
    results["decision"] = "Spurious"
    return results

# Cointegration accepted
results["cointegrated"] = True
results["decision"] = "Cointegration"

In [ ]:
# Estimate ECM
ect_lag = residuals.shift(1)
ecm_model, ecm_df = estimate_ecm(y, x, ect_lag)
results["ecm_summary"] = {
    "params": ecm_model.params.to_dict(),
    "pvalues": ecm_model.pvalues.to_dict(),
    "rsquared": ecm_model.rsquared
}

results["ecm_data"] = ecm_df
gamma = ecm_model.params.get("ect_lag", np.nan)
gamma_p = ecm_model.pvalues.get("ect_lag", np.nan)

if pd.notna(gamma):
    if gamma < 0 and gamma_p < 0.05:
        results["ecm_interpretation"] = (
            "The error-correction term is negative and significant, implying adjustment back toward equilibrium."
        )

    elif gamma < 0:
        results["ecm_interpretation"] = (
            "The error-correction term is negative but not statistically significant."
        )

    else:
        results["ecm_interpretation"] = (
            "The error-correction term is not negative, weakening the standard ECM interpretation."
        )

else:
    results["ecm_interpretation"] = "Could not interpret the error-correction term."
    
return results

# Create pairs strategy (baseline + Kalman)
- Start with the static hedge ratio (using OLS) (pairs-det.py)
- Develop the Kalman Filter script here (kalman.py)
- Create the dynamic model here and standardise to form the trading signal (pairs-det.py)

# Implementation 
- For backtest here and compare the Kalman indicator to the baseline and a simple long strategy of only Asset A and Asset B (backtest.py)